In [ ]:
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import pandas as pd
from google.colab import drive
import os
from datetime import datetime
import unicodedata
import json

In [ ]:
URL = 'http://www.diputados.gob.mx/LeyesBiblio/index.htm'
headers = {
    "User-Agent": "FESAragon-ScrapingBot/1.0",
    "Accept": "text/html,application/xhtml+xml"
}

def consumir_pagina(url, headers=None, timeout=60):
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        print(f"Petición exitosa. Código de estado: {response.status_code}")
        return response.text, True
    except requests.exceptions.RequestException as e:
        print(f"Error al consumir la página: {e}")
        return None, False

html, exito = consumir_pagina(URL, headers)

In [ ]:
response = requests.get(URL, headers=headers)

# Estructura del sitio

In [ ]:
html = response.text
print(html)

In [ ]:
soup = BeautifulSoup(html, 'lxml')
type(soup)

In [ ]:
h1 = soup.find('h1')
print(h1)

In [ ]:
divs = soup.find('div')
len(divs)

In [ ]:
tablas = soup.find('table')
tablas = tablas.find_all('table')
len(tablas)

In [ ]:
enlaces = soup.find_all('a')
print(len(enlaces))

In [ ]:
imagenes = soup.find_all('img')
print(len(imagenes))

In [ ]:
# estructura del contenido de la pagina
filas = soup.select("table tr")

count = 0
for fila in filas:
    celdas = fila.find_all(["td", "th"])
    if len(celdas) >= 4:
        texto_c0 = celdas[0].get_text(strip=True).lower()
        texto_c1 = celdas[1].get_text(strip=True).lower()
        if texto_c0 in {"no.", "nombre", "última reforma", "fecha", "ley", "ordenamiento"}:
            continue

        print(f"\n=== FILA {count+1} ===")
        print(f"CELDA 0 texto: '{celdas[0].get_text(strip=True)}'")
        print(f"CELDA 1 texto: '{celdas[1].get_text(strip=True)}'")
        print(f"CELDA 2 texto: '{celdas[2].get_text(strip=True)}'")
        print(f"CELDA 3 texto: '{celdas[3].get_text(strip=True)}'")
        print(f"CELDA 1 HTML:\n{celdas[1]}")
        print(f"CELDA 2 HTML:\n{celdas[2]}")

        count += 1
        if count >= 3:
            break


# Extracción de información

In [ ]:
def limpiar_texto(texto):
    if not texto:
        return ""
    return texto.strip()

In [ ]:
datos_leyes = []
filas = soup.select("table tr")
ENCABEZADOS = {"no.", "nombre", "última reforma", "fecha", "ley", "ordenamiento"}

for fila in filas:
    celdas = fila.find_all(["td", "th"])
    if len(celdas) < 3:
        continue

    texto_c0 = celdas[0].get_text(strip=True).lower()
    texto_c1 = celdas[1].get_text(strip=True).lower()
    if texto_c0 in ENCABEZADOS or texto_c1 in ENCABEZADOS:
        continue

    numero = limpiar_texto(celdas[0].get_text(strip=True))
    nombre_ley = ""
    fecha_publicacion = ""

    a_nombre = celdas[1].find("a")
    if a_nombre:
        nombre_ley = limpiar_texto(a_nombre.get_text(strip=True))
    for font in celdas[1].find_all("font"):
        if not font.find_parent("a"):
            texto = limpiar_texto(font.get_text(strip=True))
            if texto.startswith("DOF"):
                fecha_publicacion = texto
                break
    if not nombre_ley:
        continue

    ultima_reforma = ""
    font_reforma = celdas[2].find("font")
    if font_reforma:
        ultima_reforma = limpiar_texto(font_reforma.get_text(strip=True))


    url_pdf = None
    url_doc = None
    url_epub = None
    for a in fila.find_all("a"):
        href = a.get("href", "")
        if not href:
            continue
        enlace_completo = urljoin(URL, href)
        href_lower = href.lower()

        if ".pdf" in href_lower and url_pdf is None:
            url_pdf = enlace_completo
        elif ".doc" in href_lower and url_doc is None:
            url_doc = enlace_completo
        elif ".epub" in href_lower and url_epub is None:
            url_epub = enlace_completo

    # limpiar DOF de las fechas
    fecha_publicacion = fecha_publicacion.replace("DOF ", "").strip()
    ultima_reforma = ultima_reforma.replace("DOF ", "").strip()

    if url_pdf:
        datos_leyes.append({
            "numero": numero,
            "nombre_ley": nombre_ley,
            "fecha_publicacion": fecha_publicacion,
            "ultima_reforma": ultima_reforma,
            "url_pdf": url_pdf,
            "url_doc": url_doc,
            "url_epub": url_epub
        })

print(f"Total de leyes extraídas: {len(datos_leyes)}")

In [ ]:
if datos_leyes:
    print("LEYES:\n")

    for ley in datos_leyes[0:10]:
        print(ley)

In [ ]:
print(type(datos_leyes))

In [ ]:
print(type(datos_leyes[0]))

# DataFrame con la información

In [ ]:
df_leyes = pd.DataFrame(datos_leyes)
print("Estructura del DataFrame:")
df_leyes.info()

In [ ]:
df_leyes.head()

# Descargar archivos

In [ ]:
drive.mount('/content/drive/')

In [ ]:
fechahora = datetime.now().strftime("%Y%m%d_%H%M%S")
ruta_descarga = f"/content/drive/MyDrive/add/leyes/pdf_{fechahora}/"

os.makedirs(ruta_descarga, exist_ok=True)
print(f"Directorio listo en Google Drive:\n{ruta_descarga}")

In [ ]:
def simplificar_nombre(nombre):
    nombre = nombre.lower()
    nombre = "".join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = re.sub(r'[^a-z0-9_]+', '_', nombre)
    nombre = re.sub(r'_+', '_', nombre).strip('_')
    palabras = nombre.split('_')[:4]
    return "_".join(palabras)

In [ ]:
limite_pruebas = 5
contador = 0

print("Iniciando la descarga de documentos...\n")

for index, fila in df_leyes.iterrows():
    if limite_pruebas and contador >= limite_pruebas:
        print(f"\n[INFO] Prueba completada con éxito. Se descargaron {limite_pruebas} archivos.")
        break

    url_pdf = fila['url_pdf']
    if not url_pdf:
        continue

    num_limpio = str(fila['numero']).zfill(3)
    nombre_limpio = simplificar_nombre(fila['nombre_ley'])

    nombre_archivo = f"{num_limpio}_{nombre_limpio}.pdf"
    ruta_final_pdf = os.path.join(ruta_descarga, nombre_archivo)

    try:
        respuesta_pdf = requests.get(url_pdf, headers=headers, timeout=20)
        respuesta_pdf.raise_for_status()

        with open(ruta_final_pdf, 'wb') as archivo_pdf:
            archivo_pdf.write(respuesta_pdf.content)

        print(f"Descargado con éxito: {nombre_archivo}")
        contador += 1
        time.sleep(1)

    except Exception as e:
        print(f"Error al descargar {nombre_archivo}: {e}")

## Información procesada

In [ ]:
ruta_log = "/content/drive/MyDrive/add/leyes/datos"
os.makedirs(ruta_log, exist_ok=True)

# Definir la ruta del archivo CSV y guardar el DataFrame
archivo_csv = os.path.join(ruta_log, "leyes_scraping.csv")
df_leyes.to_csv(archivo_csv, index=False, encoding="utf-8")

log_datos = {
    "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_registros_encontrados": len(datos_leyes),
    "total_registros_dataframe": len(df_leyes),
    "ruta_almacenamiento_csv": archivo_csv,
    "ruta_almacenamiento_pdfs": ruta_descarga,
    "estatus_proceso": "Exitoso"
}

archivo_log = os.path.join(ruta_log, "log_proceso.json")

with open(archivo_log, "w", encoding="utf-8") as f:
    json.dump(log_datos, f, ensure_ascii=False, indent=4)

print("Log JSON y CSV generados con exito.")

# Logging JSON

In [ ]:
log_datos = {
    "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_registros_encontrados": len(datos_leyes),
    "total_registros_dataframe": len(df_leyes),
    "ruta_almacenamiento_csv": archivo_csv,
    "ruta_almacenamiento_pdfs": ruta_descarga,
    "estatus_proceso": "Exitoso"
}

ruta_log = "/content/drive/MyDrive/add/leyes/datos"
os.makedirs(ruta_log, exist_ok=True)
archivo_log = os.path.join(ruta_log, "log_proceso.json")

with open(archivo_log, "w", encoding="utf-8") as f:
    json.dump(log_datos, f, ensure_ascii=False, indent=4)

print("Log JSON generado con éxito.")